Notebook 3 — Clasificadores RNN

CELDA 1 — Instalación

In [2]:
!pip install yfinance ta tensorflow scikit-learn --quiet

  Preparing metadata (setup.py) ... done


CELDA 2 — Librerías

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import json

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    GRU,
    SimpleRNN,
    Dense,
    Dropout,
    Bidirectional
)

CELDA 3 — Configuración

In [4]:
ticker = "BVN"

LOOKBACK = 60

EPOCHS = 80

BATCH = 64

CELDA 4 — Descarga

In [5]:
df = yf.download(
    ticker,
    period="2y",
    progress=False
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df[["Close"]]

df.dropna(inplace=True)

df.head()

/tmp/ipykernel_2210/1424794913.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(


Price,Close
Date,
2024-06-20,16.254086
2024-06-21,16.254086
2024-06-24,16.663773
2024-06-25,16.273142
2024-06-26,15.977786


CELDA 5 — Variable BUY/SELL

In [6]:
df["Trend"] = np.where(
    df["Close"].shift(-1) > df["Close"],
    1,
    0
)

df.dropna(inplace=True)

CELDA 6 — Escalamiento

In [7]:
scaler = MinMaxScaler()

precios = scaler.fit_transform(
    df[["Close"]]
)

CELDA 7 — Ventanas temporales

In [8]:
X = []
y = []

for i in range(LOOKBACK, len(precios)-1):

    X.append(
        precios[i-LOOKBACK:i]
    )

    y.append(
        df["Trend"].iloc[i]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(440, 60, 1)
(440,)


CELDA 8 — Train/Test

In [9]:
corte = int(len(X)*0.80)

X_train = X[:corte]
X_test = X[corte:]

y_train = y[:corte]
y_test = y[corte:]

CELDA 9 — Modelo LSTM

In [10]:
modelo_lstm = Sequential()

modelo_lstm.add(
    LSTM(
        260,
        return_sequences=True,
        input_shape=(X.shape[1],1)
    )
)

modelo_lstm.add(Dropout(0.2))

modelo_lstm.add(
    LSTM(130)
)

modelo_lstm.add(
    Dense(
        1,
        activation="sigmoid"
    )
)

modelo_lstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

hist_lstm = modelo_lstm.fit(
    X_train,
    y_train,
    epochs=80,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 509ms/step - accuracy: 0.4875 - loss: 0.6952 - val_accuracy: 0.6338 - val_loss: 0.6702
Epoch 2/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 530ms/step - accuracy: 0.5267 - loss: 0.6899 - val_accuracy: 0.6338 - val_loss: 0.6749
Epoch 3/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 693ms/step - accuracy: 0.5267 - loss: 0.6907 - val_accuracy: 0.6338 - val_loss: 0.6724
Epoch 4/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 636ms/step - accuracy: 0.5267 - loss: 0.6902 - val_accuracy: 0.6338 - val_loss: 0.6727
Epoch 5/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 379ms/step - accuracy: 0.5267 - loss: 0.6899 - val_accuracy: 0.6338 - val_loss: 0.6717
Epoch 6/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 394ms/step - accuracy: 0.5267 - loss: 0.6896 - val_accuracy: 0.6338 - val_loss: 0.6771
Epoch 7/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 389ms/step - accuracy: 0.5267 - loss: 0.6884 - val_accuracy: 0.6338 - val_loss: 0.6710
Epoch 8/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 386ms/step - accuracy: 0.5267 - loss: 0.6906 - val_accuracy: 0.6338 - val_loss:

CELDA 10 — BiLSTM

In [11]:
modelo_bi = Sequential()

modelo_bi.add(
    Bidirectional(
        LSTM(
            200,
            return_sequences=True
        ),
        input_shape=(X.shape[1],1)
    )
)

modelo_bi.add(Dropout(0.3))

modelo_bi.add(
    Bidirectional(
        LSTM(100)
    )
)

modelo_bi.add(
    Dense(
        1,
        activation="sigmoid"
    )
)

modelo_bi.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

hist_bi = modelo_bi.fit(
    X_train,
    y_train,
    epochs=80,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.5267 - loss: 0.6944 - val_accuracy: 0.6338 - val_loss: 0.6652
Epoch 2/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 590ms/step - accuracy: 0.5267 - loss: 0.6897 - val_accuracy: 0.6338 - val_loss: 0.6704
Epoch 3/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 604ms/step - accuracy: 0.5231 - loss: 0.6901 - val_accuracy: 0.6338 - val_loss: 0.6703
Epoch 4/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 942ms/step - accuracy: 0.5231 - loss: 0.6899 - val_accuracy: 0.6338 - val_loss: 0.6684
Epoch 5/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 580ms/step - accuracy: 0.5267 - loss: 0.6894 - val_accuracy: 0.6338 - val_loss: 0.6686
Epoch 6/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 588ms/step - accuracy: 0.5267 - loss: 0.6896 - val_accuracy: 0.6338 - val_loss: 0.6769
Epoch 7/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 585ms/step - accuracy: 0.5267 - loss: 0.6899 - val_accuracy: 0.6338 - val_loss: 0.6745
Epoch 8/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 602ms/step - accuracy: 0.5196 - loss: 0.6889 - val_accuracy: 0.6338 - val_loss: 0

CELDA 11 — GRU

In [12]:
modelo_gru = Sequential()

modelo_gru.add(
    GRU(
        280,
        return_sequences=True,
        input_shape=(X.shape[1],1)
    )
)

modelo_gru.add(
    GRU(140)
)

modelo_gru.add(
    Dense(
        1,
        activation="sigmoid"
    )
)

modelo_gru.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

hist_gru = modelo_gru.fit(
    X_train,
    y_train,
    epochs=80,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 525ms/step - accuracy: 0.4662 - loss: 0.7014 - val_accuracy: 0.3662 - val_loss: 0.7011
Epoch 2/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 397ms/step - accuracy: 0.4982 - loss: 0.6938 - val_accuracy: 0.6338 - val_loss: 0.6738
Epoch 3/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 574ms/step - accuracy: 0.5267 - loss: 0.6904 - val_accuracy: 0.6338 - val_loss: 0.6732
Epoch 4/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 431ms/step - accuracy: 0.5267 - loss: 0.6899 - val_accuracy: 0.6338 - val_loss: 0.6740
Epoch 5/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 395ms/step - accuracy: 0.5267 - loss: 0.6897 - val_accuracy: 0.6338 - val_loss: 0.6815
Epoch 6/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 407ms/step - accuracy: 0.5374 - loss: 0.6898 - val_accuracy: 0.6338 - val_loss: 0.6890
Epoch 7/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 405ms/step - accuracy: 0.5338 - loss: 0.6897 - val_accuracy: 0.6338 - val_loss: 0.6930
Epoch 8/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 750ms/step - accuracy: 0.5480 - loss: 0.6904 - val_accuracy: 0.6338 - val_loss:

CELDA 12 — SimpleRNN

In [13]:
modelo_rnn = Sequential()

modelo_rnn.add(
    SimpleRNN(
        180,
        return_sequences=True,
        input_shape=(X.shape[1],1)
    )
)

modelo_rnn.add(
    SimpleRNN(90)
)

modelo_rnn.add(
    Dense(
        1,
        activation="sigmoid"
    )
)

modelo_rnn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

hist_rnn = modelo_rnn.fit(
    X_train,
    y_train,
    epochs=80,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 183ms/step - accuracy: 0.5267 - loss: 0.7319 - val_accuracy: 0.6338 - val_loss: 0.6708
Epoch 2/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.5018 - loss: 0.6980 - val_accuracy: 0.4930 - val_loss: 0.6933
Epoch 3/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.5267 - loss: 0.7373 - val_accuracy: 0.4225 - val_loss: 0.7352
Epoch 4/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.4698 - loss: 0.7541 - val_accuracy: 0.3662 - val_loss: 0.9749
Epoch 5/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 162ms/step - accuracy: 0.4875 - loss: 0.7702 - val_accuracy: 0.6338 - val_loss: 0.6679
Epoch 6/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step - accuracy: 0.4911 - loss: 0.7156 - val_accuracy: 0.6338 - val_loss: 0.6623
Epoch 7/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.4982 - loss: 0.7169 - val_accuracy: 0.4930 - val_loss: 0.6935
Epoch 8/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.4555 - loss: 0.7006 - val_accuracy: 0.6338 - val_loss: 0.6

CELDA 13 — Función de métricas

In [14]:
def evaluar(modelo):

    pred = modelo.predict(X_test)

    pred = (pred > 0.5).astype(int)

    return {
        "accuracy":
            accuracy_score(y_test,pred),

        "precision":
            precision_score(y_test,pred),

        "recall":
            recall_score(y_test,pred),

        "f1":
            f1_score(y_test,pred)
    }

CELDA 14 — Evaluación

In [15]:
m_lstm = evaluar(modelo_lstm)

m_bi = evaluar(modelo_bi)

m_gru = evaluar(modelo_gru)

m_rnn = evaluar(modelo_rnn)

print(m_lstm)
print(m_bi)
print(m_gru)
print(m_rnn)

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 224ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 430ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step 

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 291ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 204ms/step
{'accuracy': 0.5113636363636364, 'precision': 0.5217391304347826, 'recall': 0.5333333333333333, 'f1': 0.5274725274725275}
{'accuracy': 0.48863636363636365, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
{'accuracy': 0.5227272727272727, 'precision': 0.5230769230769231, 'recall': 0.7555555555555555, 'f1': 0.6181818181818182}
{'accuracy': 0.5113636363636364, 'precision': 0.5113636363636364, 'recall': 1.0, 'f1': 0.6766917293233082}


CELDA 15 — Mejor modelo

In [16]:
modelos = {
    "LSTM": m_lstm,
    "BiLSTM": m_bi,
    "GRU": m_gru,
    "SimpleRNN": m_rnn
}

mejor = max(
    modelos,
    key=lambda x: modelos[x]["accuracy"]
)

print("Mejor modelo:", mejor)

Mejor modelo: GRU


CELDA 16 — JSON

In [22]:
salida = {
    "ticker": ticker,

    "fechas": df.index.astype(str).tolist(),

    "precios": df["Close"].round(4).tolist(),

    "modelos": [

        {
            "id": "LSTM",
            "acc": m_lstm["accuracy"],
            "prec": m_lstm["precision"],
            "rec": m_lstm["recall"],
            "f1": m_lstm["f1"],
            "accHist": hist_lstm.history["accuracy"]
        },

        {
            "id": "BiLSTM",
            "acc": m_bi["accuracy"],
            "prec": m_bi["precision"],
            "rec": m_bi["recall"],
            "f1": m_bi["f1"],
            "accHist": hist_bi.history["accuracy"]
        },

        {
            "id": "GRU",
            "acc": m_gru["accuracy"],
            "prec": m_gru["precision"],
            "rec": m_gru["recall"],
            "f1": m_gru["f1"],
            "accHist": hist_gru.history["accuracy"]
        },

        {
            "id": "SimpleRNN",
            "acc": m_rnn["accuracy"],
            "prec": m_rnn["precision"],
            "rec": m_rnn["recall"],
            "f1": m_rnn["f1"],
            "accHist": hist_rnn.history["accuracy"]
        }
    ],

    "mejor_modelo": mejor
}

CELDA 17 — Exportación

In [25]:
with open("rnns.json", "w") as f:
    json.dump(salida, f, indent=4)

print("JSON exportado.")

JSON exportado.


CELDA 18 — Descarga

In [26]:
from google.colab import files

files.download(
    "rnns.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>